# Sesión 4 · Integración de fuentes heterogéneas
## Laboratorio autocontenido con SQLite: la tienda Conecta

**Objetivo:** construir un dataset de pedidos fiable combinando clientes de **SQLite**, pedidos de **CSV** y un catálogo **JSON anidado**. Todos los datos son ficticios y se generan aquí; no se descargan datasets ni se necesitan ficheros previos.

Al terminar podrás identificar esquemas, convertir tipos, distinguir claves técnicas y de negocio, detectar duplicados y ausencias, comprobar relaciones y unir fuentes sin multiplicar registros por error.

**Duración orientativa: 150 minutos.** Introducción y preparación (20), generación e inspección (25), limpieza (40), integración (35), análisis y puesta en común (30). Las soluciones siguen a los ejercicios: intenta resolver cada reto antes de ejecutar su solución. **«Ejecutar todo» crea las tres fuentes y completa el laboratorio sin configurar conexiones.**

### Requisitos y alcance
- Python 3.10 o posterior y Jupyter Notebook, JupyterLab o VS Code con soporte Jupyter.
- **SQLite está integrado en Python mediante `sqlite3`: no necesitas instalar un servidor, usar Docker ni configurar usuarios o contraseñas.** Se requiere SQLite 3.24 o posterior para el UPSERT; la primera celda comprueba la versión disponible.
- Se utiliza pandas >= 2.2 y < 4. La primera celda lo instala si falta; solo esa instalación requiere Internet. Si la versión instalada no es compatible, se indica cómo actualizarla y reiniciar el kernel.
- Al ejecutar se crea `datos_sesion4_sqlite/` en el directorio de trabajo del kernel: contendrá `sesion4_integracion.sqlite`, las fuentes CSV/JSON y los resultados. **Todo el código y todos los datos iniciales están incluidos en este único `.ipynb`.**
- La inicialización actualiza por clave técnica las filas de ejemplo al repetirla, sin borrar bases ni tablas. Los datos defectuosos se conservan en las fuentes originales: la limpieza se hace en copias en memoria y los resultados se exportan aparte.

Abre el cuaderno y ejecuta las celdas de arriba abajo. Si vuelves a abrirlo otro día, ejecuta todo para reconstruir también las variables en memoria. La base SQLite permanece como un fichero local que puedes inspeccionar con cualquier visor SQLite.


In [ ]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("pandas") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas>=2.2,<4"])

import re
import csv
import json
import sqlite3
from contextlib import contextmanager
from pathlib import Path
from decimal import Decimal, InvalidOperation
import pandas as pd
from IPython.display import display

version_pandas = tuple(int(parte) for parte in pd.__version__.split(".")[:2])
if not (version_pandas >= (2, 2) and version_pandas < (4, 0)):
    raise RuntimeError('Se requiere pandas >= 2.2 y < 4. Ejecuta %pip install "pandas>=2.2,<4" en otra celda y reinicia el kernel.')
if sqlite3.sqlite_version_info < (3, 24, 0):
    raise RuntimeError("Se requiere SQLite >= 3.24. Utiliza una distribución de Python con SQLite actualizado.")
pd.set_option("display.max_columns", 30)
CARPETA = Path.cwd() / "datos_sesion4_sqlite"
CARPETA.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0], "| pandas:", pd.__version__, "| SQLite:", sqlite3.sqlite_version)
print("Directorio de trabajo:", CARPETA.resolve())


### 1. El mapa de las fuentes

| Fuente | Qué representa una fila | Clave de negocio | Particularidades |
|---|---|---|---|
| SQLite · `s4_clientes_raw` | Una captura/versionado de un cliente | `cliente_id` | `fila_id` identifica físicamente cada fila; admite clientes repetidos y `NULL` |
| CSV · `pedidos_raw.csv` | Un pedido de un solo producto | `pedido_id` | Separador `;`, coma o punto decimal, fechas ISO o día/mes/año, vacíos |
| JSON · `catalogo_raw.json` | Un producto, dentro de `data.productos` | `producto_id` | Objetos anidados, números y booleanos nativos |

Relaciones esperadas tras limpiar: **cliente 1 → N pedidos** y **producto 1 → N pedidos**. El resultado tendrá una fila por pedido, no una fila por cliente. No se debe unir por nombre, posición de fila o precio.

En una API, este JSON podría ser el cuerpo de una respuesta HTTP; aquí se guarda localmente para asegurar la reproducibilidad. Los registros de aplicaciones suelen tener eventos y marcas de tiempo: antes de incorporarlos habría que definir su granularidad, su zona horaria y cómo relacionarlos con pedidos. API es un mecanismo de acceso, JSON un formato y SQLite un gestor: no son categorías equivalentes. La ampliación final propone practicar con ambos.


In [ ]:
# Estas estructuras son la semilla reproducible. No hay aleatoriedad.
columnas_clientes = ["fila_id", "cliente_id", "nombre", "email", "ciudad", "actualizado_en"]
clientes_semilla = [
    (1, "C001", "Ana López", " ANA@EJEMPLO.ES ", "madrid", "2026-10-01 09:00:00"),
    (2, "C002", "Luis Pérez", None, "Barcelona", "2026-10-01 10:00:00"),
    (3, "C003", "Marta Ruiz", "marta@ejemplo.es", " sevilla ", "2026-10-02 09:00:00"),
    (4, "C003", "Marta Ruiz", "marta@ejemplo.es", " sevilla ", "2026-10-02 09:00:00"),
    (5, " c004 ", "Diego Sol", "DIEGO@EJEMPLO.ES", None, "2026-10-03 09:00:00"),
    (6, "C005", "Elena Gil", "", "Valencia", "2026-10-01 09:00:00"),
    (7, "C005", "Elena Gil", "elena@ejemplo.es", "Valencia", "2026-10-10 09:00:00"),
    (8, "C006", "Pablo Mar", "pablo@ejemplo.es", "Málaga", "2026-10-04 09:00:00"),
    (9, "C007", None, None, "Bilbao", "2026-10-05 09:00:00"),
]

columnas_pedidos = ["pedido_id", "cliente_id", "producto_id", "cantidad", "precio_unitario", "fecha"]
pedidos_semilla = [
    ("P001", " c001 ", "pr01", "2", "19,90", "01/10/2026"),
    ("P002", "C002", "PR02", "1", "35.50", "2026-10-02"),
    ("P003", "C003", "PR03", "", "12,00", "2026-10-03"),
    ("P004", "C004", "PR01", "1", "", "04/10/2026"),
    ("P005", "C005", "PR02", "3", "35,50", "2026-10-05"),
    ("P006", "C006", "PR04", "1", "1.299,00", "06/10/2026"),
    ("P007", "C999", "PR01", "1", "19,90", "2026-10-07"),
    ("P008", "", "PR02", "2", "35,50", "2026-10-08"),
    ("P009", "C001", "PR99", "1", "9,99", "2026-10-09"),
    ("P010", "C002", "PR03", "dos", "12,00", "2026-10-10"),
    ("P011", "C003", "PR03", "2", "12,00", "31/02/2026"),
    ("P012", "C007", "PR03", "2", "12,00", "12/10/2026"),
    ("P002", "C002", "PR02", "1", "35.50", "2026-10-02"),
    ("P013", "C005", "PR02", "2", "35,50", "2026-10-13"),
    ("P014", "C006", "PR01", "1", "-5,00", "2026-10-14"),
    ("P015", "C001", "PR03", "1", "no_disponible", "2026-10-15"),
]

catalogo_semilla = {
    "meta": {"origen": "API simulada de catálogo", "version": 1, "moneda": "EUR"},
    "data": {"productos": [
        {"producto_id": "PR01", "nombre": "Ratón", "precio": 19.90, "atributos": {"categoria": "Accesorios", "activo": True}},
        {"producto_id": "PR02", "nombre": "Teclado", "precio": 35.50, "atributos": {"categoria": "Accesorios", "activo": True}},
        {"producto_id": "PR03", "nombre": "Cuaderno", "precio": 12.00, "atributos": {"categoria": "Papelería", "activo": True}},
        {"producto_id": "PR04", "nombre": "Portátil", "precio": 1299.00, "atributos": {"categoria": "Informática", "activo": True}},
        {"producto_id": "PR05", "nombre": "Bolígrafo", "precio": 7.50, "atributos": {"categoria": "Papelería", "activo": False}},
    ]},
}

with (CARPETA / "pedidos_raw.csv").open("w", encoding="utf-8", newline="") as f:
    escritor = csv.writer(f, delimiter=";")
    escritor.writerow(columnas_pedidos)
    escritor.writerows(pedidos_semilla)
with (CARPETA / "catalogo_raw.json").open("w", encoding="utf-8") as f:
    json.dump(catalogo_semilla, f, ensure_ascii=False, indent=2)
print("Generados pedidos_raw.csv (16 filas) y catalogo_raw.json (5 productos).")


### 2. Crear y cargar SQLite desde Python

La siguiente celda crea **`datos_sesion4_sqlite/sesion4_integracion.sqlite`** y la tabla de clientes utilizando SQL ejecutado desde el propio cuaderno. No se necesita ninguna aplicación adicional. `sqlite3.connect()` crea el fichero si no existe; `CREATE TABLE IF NOT EXISTS` crea las tablas y `executemany()` inserta los registros usando parámetros `?`.

`fila_id` es una clave primaria técnica. `cliente_id` no tiene restricción de unicidad en esta tabla de entrada: así podemos practicar con duplicados de negocio y varias versiones del mismo cliente. Los `None` de Python se almacenan como `NULL` SQL. En SQLite guardamos las fechas como **texto ISO** y las convertimos a fechas de pandas durante la limpieza; SQLite no tiene un tipo de almacenamiento nativo para fechas.

La rutina rechaza una base existente con tablas que no contenga el marcador de este laboratorio. Al repetirla, `ON CONFLICT(fila_id) DO UPDATE` actualiza las nueve filas de semilla; no elimina filas nuevas añadidas por el alumnado. Para recuperar el caso original después de añadir registros, cambia `RUTA_BD` por otro nombre de fichero dentro de la carpeta de prácticas y vuelve a ejecutar todo. Las comprobaciones con cifras fijas presuponen la semilla original.

La función `conectar()` confirma la transacción si termina correctamente, la revierte si falla y cierra la conexión al salir del bloque. La base queda guardada en disco.


In [ ]:
RUTA_BD = CARPETA / "sesion4_integracion.sqlite"

@contextmanager
def conectar():
    conn = sqlite3.connect(RUTA_BD, timeout=10)
    conn.row_factory = sqlite3.Row
    try:
        with conn:  # commit al terminar; rollback si se produce una excepción
            yield conn
    finally:
        conn.close()

with conectar() as conn:
    tablas = {fila["name"] for fila in conn.execute(
        "SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%'"
    )}
    if tablas and "s4_laboratorio_meta" not in tablas:
        raise RuntimeError("Base existente ajena al laboratorio. Elige otro fichero RUTA_BD.")
    if "s4_laboratorio_meta" in tablas:
        marca = conn.execute("SELECT version FROM s4_laboratorio_meta WHERE id = 1").fetchone()
        if marca is None or marca["version"] != "conecta_sqlite_v1":
            raise RuntimeError("Marcador incompatible. Elige otro fichero de prácticas.")
    conn.execute("CREATE TABLE IF NOT EXISTS s4_laboratorio_meta (id INTEGER PRIMARY KEY, version TEXT NOT NULL)")
    conn.execute("INSERT OR IGNORE INTO s4_laboratorio_meta VALUES (1, 'conecta_sqlite_v1')")
    conn.execute("""
        CREATE TABLE IF NOT EXISTS s4_clientes_raw (
            fila_id INTEGER PRIMARY KEY,
            cliente_id TEXT, nombre TEXT, email TEXT,
            ciudad TEXT, actualizado_en TEXT
        )
    """)
    conn.executemany("""
        INSERT INTO s4_clientes_raw
        (fila_id, cliente_id, nombre, email, ciudad, actualizado_en)
        VALUES (?, ?, ?, ?, ?, ?)
        ON CONFLICT(fila_id) DO UPDATE SET
            cliente_id=excluded.cliente_id, nombre=excluded.nombre,
            email=excluded.email, ciudad=excluded.ciudad,
            actualizado_en=excluded.actualizado_en
    """, clientes_semilla)
    total_clientes_bd = conn.execute("SELECT COUNT(*) FROM s4_clientes_raw").fetchone()[0]

assert RUTA_BD.is_file()
print("Base creada/cargada:", RUTA_BD.resolve())
print("Filas almacenadas:", total_clientes_bd)


**Reto 1 · inspección (10 min).** Antes de limpiar: ¿qué tipos aparecen en cada origen? ¿Qué diferencia hay entre `fila_id` y `cliente_id`? ¿Cuántas filas duplicadas hay si ignoras la clave técnica? ¿Cuántos clientes distintos hay? Localiza dos `NULL` en SQLite y dos vacíos en el CSV. ¿Por qué un `NULL` SQL no es el texto `"NULL"`?


In [ ]:
def consultar_sqlite(sql, parametros=()):
    with conectar() as conn:
        return pd.read_sql_query(sql, conn, params=parametros)

# PRAGMA table_info muestra columnas, tipos declarados y claves.
display(consultar_sqlite("PRAGMA table_info(s4_clientes_raw)"))
display(consultar_sqlite("SELECT * FROM s4_clientes_raw WHERE email IS NULL OR ciudad IS NULL ORDER BY fila_id"))
display(consultar_sqlite("""
    SELECT UPPER(TRIM(cliente_id)) AS cliente_id, COUNT(*) AS capturas
    FROM s4_clientes_raw GROUP BY UPPER(TRIM(cliente_id)) HAVING COUNT(*) > 1
"""))
display(consultar_sqlite(
    "SELECT fila_id, typeof(cliente_id) AS tipo_clave, typeof(actualizado_en) AS tipo_fecha, typeof(email) AS tipo_email FROM s4_clientes_raw ORDER BY fila_id"
))

clientes_raw = consultar_sqlite("SELECT * FROM s4_clientes_raw ORDER BY fila_id")
# Leemos texto para que pandas no convierta identificadores o vacíos por nosotros.
pedidos_raw = pd.read_csv(CARPETA / "pedidos_raw.csv", sep=";", dtype="string", keep_default_na=False)
pedidos_raw["fila_csv"] = range(2, len(pedidos_raw) + 2)  # línea física, contando cabecera
with (CARPETA / "catalogo_raw.json").open(encoding="utf-8") as f:
    respuesta_catalogo = json.load(f)
productos_raw = pd.json_normalize(respuesta_catalogo["data"]["productos"])

for nombre, tabla in {"SQLite": clientes_raw, "CSV": pedidos_raw, "JSON": productos_raw}.items():
    print("\n", nombre, "| filas:", len(tabla))
    display(tabla.head(10))
    display(tabla.dtypes.astype(str).rename("tipo observado").to_frame())
print("Una celda vacía del CSV todavía es texto vacío:", repr(pedidos_raw.loc[2, "cantidad"]))


### 3. Contrato de calidad y conversiones

Decidimos las reglas **antes** de unir:

| Campo o problema | Regla del laboratorio |
|---|---|
| Identificadores | Texto, sin espacios exteriores, en mayúsculas. Las claves de dimensiones no pueden faltar ni repetirse |
| Cliente duplicado idéntico | Conservar una copia; `fila_id` no forma parte del contenido de negocio |
| Varias versiones del cliente | Conservar la más reciente por `actualizado_en`; en empate, mayor `fila_id` |
| Nombre/ciudad ausentes | Etiqueta explícita «Nombre no informado» / «Ciudad no informada». Mantener banderas de ausencia |
| Email ausente | Mantener ausente: no inventar una dirección |
| Pedido repetido idéntico | Conservar la primera línea. Un mismo ID con contenidos diferentes exige revisión, no descarte arbitrario |
| Cantidad | Entero positivo. Si falta o es inválida, apartar el pedido; no suponer que vale 1 |
| Fechas | Solo `AAAA-MM-DD` o `DD/MM/AAAA`; una fecha imposible se convierte en `NaT` y se aparta |
| Importes | EUR, máximo dos decimales; admitir `35.50`, `35,50` y miles españoles `1.299,00`; usar céntimos enteros |
| Precio vacío | Recuperarlo del catálogo **solo para esta práctica**, que asume mismo precio vigente y sin descuentos |
| Precio mal formado o negativo | Apartar el pedido. No confundir un fallo de conversión con un vacío |
| Clave sin correspondencia | Mantener en el diagnóstico mediante `left join`; apartar del dataset listo para analizar |

En un caso real, el precio histórico de venta puede diferir del catálogo actual: habría que buscar una tarifa con fecha de vigencia o revisar el pedido. Tampoco conviene imputar cantidades con la media sin una justificación del dominio.

**Reto 2 · perfilado y conversión (15 min).** Escribe una función que trate `None`, espacios y cadenas vacías de forma uniforme. Convierte precios y fechas sin perder información sobre los errores. ¿Por qué es peligroso quitar todos los puntos de cualquier importe? ¿Qué pasaría con `35.50`?


In [ ]:
def texto_limpio(serie):
    return serie.astype("string").str.strip().replace("", pd.NA)

def clave(serie):
    return texto_limpio(serie).str.upper()

def euros_a_centimos(valor):
    """Contrato explícito: decimal europeo o decimal con punto; sin redondeo silencioso."""
    if pd.isna(valor) or str(valor).strip() == "":
        return pd.NA
    texto = str(valor).strip()
    europeo = r"-?(?:[0-9]+|[0-9]{1,3}(?:\.[0-9]{3})+),[0-9]{1,2}"
    decimal_punto = r"-?[0-9]+(?:\.[0-9]{1,2})?"
    if re.fullmatch(europeo, texto):
        texto = texto.replace(".", "").replace(",", ".")
    elif not re.fullmatch(decimal_punto, texto):
        return pd.NA
    try:
        return int(Decimal(texto) * 100)
    except (InvalidOperation, ValueError):
        return pd.NA

def perfil(tabla):
    normalizada = tabla.copy()
    for c in normalizada.select_dtypes(include=["object", "string"]).columns:
        normalizada[c] = texto_limpio(normalizada[c])
    return pd.DataFrame({"tipo": tabla.dtypes.astype(str),
                         "ausentes_incluyendo_vacios": normalizada.isna().sum(),
                         "distintos_sin_nulos": normalizada.nunique(dropna=True)})

display(perfil(clientes_raw))
display(perfil(pedidos_raw))
assert euros_a_centimos("35.50") == 3550
assert euros_a_centimos("1.299,00") == 129900
assert pd.isna(euros_a_centimos("no_disponible"))
assert pd.isna(euros_a_centimos("1.299"))  # ambiguo: fuera del contrato


**Reto 3 · limpiar clientes (15 min).** Separa duplicados idénticos de versiones con información distinta. Antes de decidir qué versión queda, muestra las filas implicadas. Conserva indicadores para saber qué valores fueron rellenados. Resultado esperado: **7 clientes**, una fila por `cliente_id`.


In [ ]:
clientes = clientes_raw.copy()
clientes["cliente_id"] = clave(clientes["cliente_id"])
for col in ["nombre", "email", "ciudad"]:
    clientes[col] = texto_limpio(clientes[col])
clientes["email"] = clientes["email"].str.lower()
clientes["ciudad"] = clientes["ciudad"].str.title()
clientes["actualizado_en"] = pd.to_datetime(clientes["actualizado_en"], errors="raise")

contenido = [c for c in clientes.columns if c != "fila_id"]
mask_duplicado = clientes.duplicated(subset=contenido, keep="first")
clientes_duplicados = clientes.loc[mask_duplicado].copy()
sin_identicos = clientes.loc[~mask_duplicado].copy()
print("Duplicados idénticos (ignorando fila_id):", len(clientes_duplicados))
display(clientes_duplicados)
print("Versiones distintas para una misma clave:")
display(sin_identicos.loc[sin_identicos.duplicated("cliente_id", keep=False)])

ordenados = sin_identicos.sort_values(["actualizado_en", "fila_id"], kind="stable")
clientes_versiones_descartadas = ordenados.loc[ordenados.duplicated("cliente_id", keep="last")].copy()
clientes = ordenados.drop_duplicates("cliente_id", keep="last").copy()
for col, etiqueta in [("nombre", "Nombre no informado"), ("ciudad", "Ciudad no informada")]:
    clientes[f"{col}_ausente"] = clientes[col].isna()
    clientes[col] = clientes[col].fillna(etiqueta)
clientes["email_ausente"] = clientes["email"].isna()

assert clientes["cliente_id"].notna().all() and clientes["cliente_id"].is_unique
assert len(clientes) == 7
display(clientes.sort_values("cliente_id"))


**Reto 4 · limpiar CSV y JSON (10 min).** Deduplica sin incluir `fila_csv` en la comparación. Después distingue cantidad vacía de cantidad incorrecta y precio vacío de precio mal escrito. Aplana el JSON y renombra los campos que podrían confundirse con los del cliente. Resultado: **15 pedidos distintos** y **5 productos**.


In [ ]:
pedidos = pedidos_raw.copy()
for col in columnas_pedidos:
    pedidos[col] = texto_limpio(pedidos[col])
for col in ["pedido_id", "cliente_id", "producto_id"]:
    pedidos[col] = clave(pedidos[col])
duplicados_csv = pedidos.loc[pedidos.duplicated(columnas_pedidos, keep="first")].copy()
pedidos = pedidos.drop_duplicates(columnas_pedidos, keep="first").copy()
if pedidos["pedido_id"].duplicated().any():
    raise ValueError("Hay pedidos con el mismo ID y contenido distinto: resuélvelos antes de continuar.")

pedidos["cantidad_ausente"] = pedidos["cantidad"].isna()
numero = pd.to_numeric(pedidos["cantidad"], errors="coerce")
es_entero_positivo = numero.notna() & numero.gt(0) & numero.mod(1).eq(0)
pedidos["cantidad_num"] = numero.where(es_entero_positivo).astype("Int64")
pedidos["precio_ausente"] = pedidos["precio_unitario"].isna()
pedidos["precio_centimos"] = pedidos["precio_unitario"].map(euros_a_centimos).astype("Int64")
pedidos["precio_malformado"] = ~pedidos["precio_ausente"] & pedidos["precio_centimos"].isna()
fecha_iso = pd.to_datetime(pedidos["fecha"], format="%Y-%m-%d", errors="coerce")
fecha_es = pd.to_datetime(pedidos["fecha"], format="%d/%m/%Y", errors="coerce")
pedidos["fecha_dt"] = fecha_iso.fillna(fecha_es)

productos = productos_raw.rename(columns={
    "nombre": "producto_nombre", "precio": "precio_catalogo_eur",
    "atributos.categoria": "categoria", "atributos.activo": "producto_activo",
}).copy()
productos["producto_id"] = clave(productos["producto_id"])
productos["catalogo_centimos"] = productos["precio_catalogo_eur"].map(euros_a_centimos).astype("Int64")
assert productos["producto_id"].notna().all() and productos["producto_id"].is_unique
assert productos["catalogo_centimos"].notna().all() and productos["catalogo_centimos"].ge(0).all()
assert len(pedidos) == 15 and len(duplicados_csv) == 1 and len(productos) == 5
display(duplicados_csv)
display(pedidos[["pedido_id", "cantidad", "cantidad_num", "precio_unitario", "precio_centimos", "fecha", "fecha_dt"]])
display(productos)


### 4. Relaciones y uniones con control de cardinalidad

**Reto 5 (10 min).** Une los pedidos sin limpiar con los clientes sin deduplicar, normalizando solo las claves. ¿Aumenta el número de filas? ¿Por qué `drop_duplicates()` después de unir puede esconder un error? Prueba `validate="many_to_one"` y explica el fallo. Después trabaja con las tablas limpias.

Una clave primaria técnica impide repetir `fila_id`, pero no impide dos capturas de `C003`. Por eso el modelo de entrada permite el problema deliberadamente. Una tabla de clientes depurada sí debería tener una restricción única en `cliente_id`. En pandas, los nulos de ambas claves pueden emparejarse entre sí; evitamos ese efecto exigiendo claves no nulas en las dimensiones.


In [ ]:
pedidos_sin_limpiar = pedidos_raw.copy()
pedidos_sin_limpiar["cliente_id"] = clave(pedidos_sin_limpiar["cliente_id"])
clientes_sin_limpiar = clientes_raw.copy()
clientes_sin_limpiar["cliente_id"] = clave(clientes_sin_limpiar["cliente_id"])
union_ingenua = pedidos_sin_limpiar.merge(clientes_sin_limpiar, on="cliente_id", how="left")
print("Pedidos de entrada:", len(pedidos_sin_limpiar), "| filas tras unión ingenua:", len(union_ingenua))
try:
    pedidos_sin_limpiar.merge(clientes_sin_limpiar, on="cliente_id", how="left", validate="many_to_one")
except pd.errors.MergeError as exc:
    print("Control de cardinalidad: error esperado →", exc)


In [ ]:
# LEFT conserva el pedido para diagnosticarlo. Cada dimensión aporta como máximo una fila.
integracion = pedidos.merge(
    clientes, on="cliente_id", how="left", validate="many_to_one", indicator="rel_cliente"
).merge(
    productos, on="producto_id", how="left", validate="many_to_one", indicator="rel_producto"
)
assert len(integracion) == len(pedidos)

integracion["precio_imputado"] = integracion["precio_ausente"] & integracion["catalogo_centimos"].notna()
integracion["precio_final_centimos"] = integracion["precio_centimos"].mask(
    integracion["precio_imputado"], integracion["catalogo_centimos"]
).astype("Int64")
integracion["difiere_del_catalogo"] = (
    integracion["precio_centimos"].notna() & integracion["catalogo_centimos"].notna()
    & integracion["precio_centimos"].ne(integracion["catalogo_centimos"])
).fillna(False)

# Una fila puede tener varios motivos. Los diagnósticos no son categorías excluyentes.
reglas = pd.DataFrame({
    "pedido_id_ausente": integracion["pedido_id"].isna(),
    "cliente_id_ausente": integracion["cliente_id"].isna(),
    "cliente_sin_correspondencia": integracion["cliente_id"].notna() & integracion["rel_cliente"].eq("left_only"),
    "producto_id_ausente": integracion["producto_id"].isna(),
    "producto_sin_correspondencia": integracion["producto_id"].notna() & integracion["rel_producto"].eq("left_only"),
    "cantidad_ausente": integracion["cantidad_ausente"],
    "cantidad_invalida": ~integracion["cantidad_ausente"] & integracion["cantidad_num"].isna(),
    "fecha_invalida_o_ausente": integracion["fecha_dt"].isna(),
    "precio_malformado": integracion["precio_malformado"],
    "precio_ausente_sin_recuperar": integracion["precio_ausente"] & integracion["precio_final_centimos"].isna(),
    "precio_negativo": integracion["precio_final_centimos"].lt(0).fillna(False),
}).fillna(False).astype(bool)
integracion["motivos_revision"] = reglas.apply(lambda fila: " | ".join(fila.index[fila]), axis=1)
rechazados = integracion.loc[reglas.any(axis=1)].copy()
validos = integracion.loc[~reglas.any(axis=1)].copy()
validos["importe_centimos"] = validos["cantidad_num"] * validos["precio_final_centimos"]

print("Relaciones de clientes:")
display(integracion["rel_cliente"].value_counts())
print("Pedidos apartados para revisión (no se borran):")
display(rechazados[["fila_csv", "pedido_id", "cliente_id", "producto_id", "motivos_revision"]])
print("Incidencias por motivo (un pedido puede aparecer en varios):")
display(reglas.sum().loc[lambda s: s.gt(0)].rename("pedidos"))


**Reto 6 · comparar joins (10 min).** Predice el resultado de `inner`, `left` y `outer`. ¿Un `inner` garantiza datos limpios? ¿Qué información se pierde al descartar automáticamente los pedidos sin cliente? Localiza el producto que no aparece en ningún pedido.


In [ ]:
comparacion = []
for modo in ["inner", "left", "outer"]:
    prueba = pedidos.merge(clientes[["cliente_id"]], on="cliente_id", how=modo,
                           validate="many_to_one", indicator=True)
    conteos = prueba["_merge"].value_counts()
    comparacion.append({"join": modo, "filas": len(prueba),
                        **{k: int(conteos.get(k, 0)) for k in ["both", "left_only", "right_only"]}})
display(pd.DataFrame(comparacion))
uso_catalogo = productos.merge(pedidos[["producto_id"]].drop_duplicates(),
                              on="producto_id", how="left", validate="one_to_one", indicator=True)
display(uso_catalogo.loc[uso_catalogo["_merge"].eq("left_only"), ["producto_id", "producto_nombre"]])
print("INNER solo comprueba coincidencia de clave: no valida fecha, cantidad ni precio.")


### 5. Dataset integrado, trazabilidad y controles

Conservamos la línea CSV de origen, las banderas de ausencia y la imputación de precio. Los importes se almacenan en **céntimos enteros** para poder sumar dinero sin errores de representación binaria. Las columnas en euros se añaden para lectura humana; los controles utilizan céntimos.

**Reto 7 (10 min).** Construye el dataset final y demuestra que cada pedido de entrada se puede explicar: o bien era duplicado, o bien queda integrado, o bien está apartado para revisión. Calcula ventas por categoría y responde si el total representa todos los pedidos originales o solo los que cumplen el contrato.


In [ ]:
dataset = validos[[
    "pedido_id", "fecha_dt", "cliente_id", "nombre", "email", "ciudad",
    "nombre_ausente", "email_ausente", "ciudad_ausente",
    "producto_id", "producto_nombre", "categoria", "cantidad_num",
    "precio_final_centimos", "importe_centimos", "precio_imputado", "difiere_del_catalogo", "fila_csv",
]].rename(columns={"fecha_dt": "fecha", "nombre": "cliente_nombre", "cantidad_num": "cantidad"})
dataset = dataset.sort_values("pedido_id").reset_index(drop=True)
dataset["precio_unitario_eur"] = dataset["precio_final_centimos"] / 100
dataset["importe_eur"] = dataset["importe_centimos"] / 100

assert dataset["pedido_id"].notna().all() and dataset["pedido_id"].is_unique
assert dataset[["cliente_id", "producto_id", "fecha", "cantidad", "precio_final_centimos"]].notna().all().all()
assert dataset["cantidad"].gt(0).all() and dataset["precio_final_centimos"].ge(0).all()
assert set(dataset["cliente_id"]) <= set(clientes["cliente_id"])
assert set(dataset["producto_id"]) <= set(productos["producto_id"])
assert len(pedidos_raw) == len(duplicados_csv) + len(rechazados) + len(dataset)
assert len(clientes_raw) == len(clientes_duplicados) + len(clientes_versiones_descartadas) + len(clientes)
assert set(dataset["pedido_id"]).isdisjoint(set(rechazados["pedido_id"]))
assert len(dataset) == 7 and len(rechazados) == 8
assert int(dataset["importe_centimos"].sum()) == 159570
assert int(dataset["precio_imputado"].sum()) == 1
assert dataset.loc[dataset["pedido_id"].eq("P004"), "precio_final_centimos"].iloc[0] == 1990
assert dataset.loc[dataset["pedido_id"].eq("P012"), "nombre_ausente"].iloc[0]

display(dataset)
resumen = dataset.groupby("categoria", as_index=False).agg(
    pedidos=("pedido_id", "count"), unidades=("cantidad", "sum"), ventas_centimos=("importe_centimos", "sum")
)
resumen["ventas_eur"] = resumen["ventas_centimos"] / 100
display(resumen)
print("Todas las comprobaciones superadas. Total válido: 1.595,70 EUR.")


In [ ]:
auditoria = {
    "version_datos": "conecta_sqlite_v1",
    "clientes_sqlite_raw": len(clientes_raw), "clientes_limpios": len(clientes),
    "clientes_duplicados_identicos": len(clientes_duplicados),
    "versiones_cliente_descartadas": len(clientes_versiones_descartadas),
    "pedidos_csv_raw": len(pedidos_raw), "pedidos_duplicados": len(duplicados_csv),
    "pedidos_revision": len(rechazados), "pedidos_integrados": len(dataset),
    "precios_imputados": int(dataset["precio_imputado"].sum()),
    "importe_valido_centimos": int(dataset["importe_centimos"].sum()),
}
dataset.to_csv(CARPETA / "dataset_integrado.csv", sep=";", index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
dataset.to_json(CARPETA / "dataset_integrado.json", orient="records", force_ascii=False, date_format="iso", indent=2)
rechazados.to_csv(CARPETA / "pedidos_revision.csv", sep=";", index=False, encoding="utf-8-sig")
duplicados_csv.to_csv(CARPETA / "pedidos_duplicados.csv", sep=";", index=False, encoding="utf-8-sig")
clientes_duplicados.to_csv(CARPETA / "clientes_duplicados.csv", sep=";", index=False, encoding="utf-8-sig")
clientes_versiones_descartadas.to_csv(CARPETA / "clientes_versiones_descartadas.csv", sep=";", index=False, encoding="utf-8-sig")
with (CARPETA / "auditoria.json").open("w", encoding="utf-8") as f:
    json.dump(auditoria, f, ensure_ascii=False, indent=2)

# CSV no conserva esquema: especificamos los tipos al recuperar el resultado.
recuperado = pd.read_csv(CARPETA / "dataset_integrado.csv", sep=";", dtype={
    "pedido_id": "string", "cliente_id": "string", "producto_id": "string",
    "cantidad": "Int64", "precio_final_centimos": "Int64", "importe_centimos": "Int64",
}, parse_dates=["fecha"])
assert len(recuperado) == 7 and int(recuperado["importe_centimos"].sum()) == 159570
display(pd.Series(auditoria, name="valor").to_frame())
print("Archivos disponibles en:", CARPETA.resolve())
for fichero in sorted(CARPETA.iterdir()):
    if fichero.is_file():
        print(" •", fichero.name)


### 6. Comprobación del aprendizaje y ampliaciones

**Resultados de referencia de la semilla original:**
- SQLite: 9 capturas → 1 duplicado idéntico y 1 versión anterior descartados → 7 clientes.
- CSV: 16 filas → 1 duplicado descartado → 15 pedidos evaluados → 7 válidos y 8 para revisión.
- JSON: 5 productos; `PR05` no aparece en los pedidos.
- Unión ingenua: 20 filas a partir de 16 pedidos, por los clientes repetidos.
- Pedido `P004`: precio vacío recuperado del catálogo; `P012`: conservado con nombre y email ausentes.
- Total de los **pedidos válidos**: **159.570 céntimos = 1.595,70 EUR**. No representa toda la facturación original: hay pedidos pendientes de revisión.

**Entrega propuesta:** el dataset integrado, el informe de auditoría y una explicación breve de las reglas adoptadas. Justifica por qué `P003`, `P010` y `P015` no se han rellenado automáticamente. Explica cómo puede sesgar el análisis excluir pedidos con incidencias.

**Ampliaciones opcionales (no forman parte de «Ejecutar todo»):**
1. Corrige una cantidad de la semilla y vuelve a generar las fuentes. Adapta los controles de referencia y explica el cambio del total. Los controles con cifras fijas detectan alteraciones del caso, no son reglas universales.
2. Añade al JSON un producto con clave repetida y verifica que se detiene la integración antes del join. Añade luego un precio CSV como `35.50` y comprueba que no se interpreta como `3550` euros.
3. Define un pequeño registro de eventos con `evento_id`, `pedido_id`, `timestamp_utc` y `estado`. Crea dos eventos para el mismo pedido. Antes de unir, conserva el último estado por pedido; compara el resultado con unir todos los eventos y explica el cambio de granularidad.
4. Sirve el catálogo mediante `python -m http.server 8000 --directory datos_sesion4_sqlite` desde una terminal situada junto al cuaderno. En una celda nueva, lee `http://127.0.0.1:8000/catalogo_raw.json` con `urllib.request.urlopen(..., timeout=5)` y `json.load`. Comprueba que el esquema es idéntico. Es una simulación HTTP local; una API real añade paginación, autenticación, límites de uso y gestión de errores.
5. Esboza el modelo limpio en SQL: clave primaria de cliente y producto, clave primaria de pedido y claves foráneas desde pedidos. ¿Por qué no se impusieron esas restricciones a la zona de entrada con datos defectuosos?

### Resolución de problemas
- **`database is locked`:** cierra las transacciones pendientes en otros programas que tengan abierta la base; vuelve a ejecutar la celda. Las conexiones del cuaderno se cierran después de cada uso.
- **No se puede abrir el fichero o directorio sin permisos:** comprueba que puedes escribir en el directorio del kernel. Puedes cambiar `CARPETA` por otra carpeta de prácticas con permiso de escritura y volver a ejecutar todo.
- **Falta pandas o su versión es incompatible:** revisa Internet/proxy si se necesita instalarlo. Para actualizar, ejecuta `%pip install "pandas>=2.2,<4"` en una celda y reinicia el kernel.
- **Una comprobación falla tras modificar el caso:** revisa las reglas y los resultados esperados. Para partir de una base nueva cambia el nombre del fichero `RUTA_BD`, sin borrar otras bases.
- **Reinicio:** «Ejecutar todo» regenera CSV y JSON, actualiza las filas de semilla de SQLite y reconstruye los resultados. No necesitas volver a configurar ninguna conexión.

### Documentación de apoyo
- [Python: SQLite, parámetros y transacciones](https://docs.python.org/3/library/sqlite3.html)
- [SQLite: tipos de datos y representación de fechas](https://www.sqlite.org/datatype3.html)
- [pandas: uniones y control de cardinalidad](https://pandas.pydata.org/docs/user_guide/merging.html)
- [pandas: conversión de fechas](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)
